# ASR vs. ASR-I — real white-box PGD comparison

This notebook reuses your **own engine** (`ids_engine2.py` / `ids_engine.py`) to
train M1 and run the masked white-box PGD attack, then reports **both** the
conventional **ASR** (malicious-evasion rate) and the proposed **ASR-I** on the
**same real predictions** — no synthetic data.

It also adds the decisive **eps = 0 (no attack)** point, where ASR equals the
clean miss rate (`1 - recall`) while ASR-I is exactly 0 by construction.

**How to run**
1. Make sure the engine `.py` files exist next to this notebook (run the
   `%%writefile` cell of your white-box notebooks, or set the paths in the
   config cell below).
2. Set the CSV paths and `N_RUNS_FIG` (use 3 for a quick check, 30 for the paper).
3. *Kernel → Restart & Run All*. Outputs: `asr_vs_asri.pdf/.png`,
   `asr_vs_asri_table.tex`, and `asr_vs_asri_runs.csv` (raw per-run numbers).

In [ ]:
# === Imports & configuration ===========================================
%matplotlib inline
import os, gc, types, importlib.util
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# Number of MCCV runs used to build the figure (averages over runs).
# 3 = quick real check; set to 30 to match the paper.
N_RUNS_FIG = 3

# One entry per dataset. `engine` is the FILE PATH to that dataset's white-box
# engine .py; `csv` is the raw dataset CSV. Datasets whose engine OR csv is
# missing are skipped automatically (so you can run HIKARI alone first).
DATASETS = {
    "HIKARI-2021": dict(
        engine="ids_engine2.py",
        csv="/home/utilizador/Doutorado/datasets/HIKARI-2021/ALLFLOWMETER_HIKARI2021.csv",
    ),
    "CIRA-CIC-DoHBrw-2020": dict(
        engine="ids_engine.py",
        csv="/home/utilizador/Doutorado/datasets/CIRA-CIC-DoHBrw-2020/CSVs/Total_CSVs/TUDAO/CIRA-CIC-DoHBrw-2020.csv",
    ),
}

In [ ]:
# === Engine loading & data preparation =================================
def load_engine(ref):
    """Load an engine as a FRESH module from a FILE PATH (bypasses sys.modules
    cache and any stale ids_engine*.py). Also accepts a module object/name."""
    if isinstance(ref, types.ModuleType):
        return ref
    if isinstance(ref, str) and os.path.exists(ref):
        name = "engmod_" + os.path.splitext(os.path.basename(ref))[0]
        spec = importlib.util.spec_from_file_location(name, ref)
        mod = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(mod)
        return mod
    return importlib.import_module(ref)


def calculate_ASR(y_true, y_adv_proba, threshold):
    """Conventional ASR over the malicious class (NO conditioning on clean
    correctness). At eps=0 (adv == clean) this equals the clean miss rate."""
    y_true = np.asarray(y_true).reshape(-1)
    y_adv = (np.asarray(y_adv_proba).reshape(-1) > threshold).astype("int32")
    malicious = (y_true == 1)
    n_mal = int(malicious.sum())
    if n_mal == 0:
        return np.nan
    return float(((malicious) & (y_adv == 0)).sum() / n_mal)


def get_prep_functions(eng):
    """Find the engine's feature-select and mask-build functions by name pattern
    (works for both the HIKARI and CIRA engines)."""
    sel = msk = None
    for n in dir(eng):
        if n.startswith("select_") and n.endswith("_features_and_target"):
            sel = getattr(eng, n)
        if n.startswith("build_") and "plausibility_mask" in n:
            msk = getattr(eng, n)
    if sel is None or msk is None:
        names = [n for n in dir(eng) if not n.startswith("_")]
        raise AttributeError(
            "Engine lacks select_*_features_and_target / build_*_plausibility_mask. "
            "Point the engine path at the correct white-box .py. Names: " + str(names))
    return sel, msk


def prepare_dataset(eng, cfg):
    """Mirror the engine's main() preprocessing on the raw CSV and return
    (X, y, feature_mask_1d). Identical cleaning to engine.main()."""
    df = pd.read_csv(cfg["csv"])
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df.dropna(inplace=True)
    df.drop_duplicates(inplace=True)
    sel, msk = get_prep_functions(eng)
    X, y, cols = sel(df)              # CIRA engine also drops ids and encodes target
    mask_1d = msk(cols)[0]            # build_*_plausibility_mask -> (mask_1d, mask_series)
    return X, y, np.asarray(mask_1d, dtype="float32").reshape(-1)

In [ ]:
# === One real white-box run (re-implements run_single_experiment) =====
def run_one(eng, X, y, mask_1d, seed):
    """Train M1, tune the threshold, run masked PGD per epsilon, and record the
    conventional ASR + ASR-I from the SAME real predictions. Adds the eps=0
    (no-attack) point. Returns [{eps, asr, asr_i}, ...]."""
    tf.keras.backend.clear_session(); gc.collect()
    eng.set_seed(seed)

    # Same splits/SMOTE/scaling/2D-reshape as the engine (8-tuple return).
    X_tr, X_va, X_te, y_tr, y_va, y_te, size, _ = eng.prepare_splits(X, y, seed)
    mask_2d = eng.reshape_to_2d(np.array([mask_1d], dtype="float32"), size)[0:1]

    train_ds = eng.make_tf_dataset(X_tr, y_tr, eng.BATCH_SIZE, shuffle=True, seed=seed)
    val_ds   = eng.make_tf_dataset(X_va, y_va, eng.BATCH_SIZE, shuffle=False, seed=seed)
    callbacks = [
        EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5, min_lr=1e-6),
    ]
    model = eng.build_mtl_model_M1(input_shape=(size, size, 1))
    model.fit(train_ds, validation_data=val_ds, epochs=eng.EPOCHS,
              callbacks=callbacks, verbose=0)

    # Threshold tuned on validation; clean probabilities on the test set.
    y_va_proba = model.predict(X_va.astype("float32"), batch_size=eng.ADV_BATCH_SIZE, verbose=0)
    t = eng.get_best_threshold(y_va, y_va_proba)
    y_clean_proba = model.predict(X_te.astype("float32"), batch_size=eng.ADV_BATCH_SIZE, verbose=0)
    logit_model = tf.keras.Model(inputs=model.input, outputs=model.get_layer("logits").output)

    # eps = 0: no attack -> ASR = clean miss rate, ASR-I = 0.
    rows = [dict(eps=0.0, asr=calculate_ASR(y_te, y_clean_proba, t), asr_i=0.0)]

    # Masked PGD per epsilon (same hyper-parameters as evaluate_adversarial).
    for eps in eng.EPSILONS:
        X_pgd = eng.pgd_attack_batched(logit_model, X_te, y_te, eps, eps / 4, 10,
                                       mask_2d, eng.ADV_BATCH_SIZE)
        y_adv_proba = model.predict(X_pgd, batch_size=eng.ADV_BATCH_SIZE, verbose=0)
        rows.append(dict(
            eps=float(eps),
            asr=calculate_ASR(y_te, y_adv_proba, t),
            asr_i=float(eng.calculate_ASR_I(y_te, y_clean_proba, y_adv_proba, t)),
        ))
        del X_pgd, y_adv_proba; gc.collect()

    del model, logit_model; gc.collect(); tf.keras.backend.clear_session()
    return rows

In [ ]:
# === Aggregation, figure, LaTeX table =================================
def build_dataframes(long_df):
    """Average ASR / ASR-I over runs, per (dataset, eps)."""
    out = {}
    for name, g in long_df.groupby("dataset"):
        agg = g.groupby("eps", as_index=False).agg(ASR=("asr", "mean"), ASR_I=("asr_i", "mean"))
        agg["gap"] = agg["ASR"] - agg["ASR_I"]
        out[name] = agg.sort_values("eps").reset_index(drop=True)
    return out


def make_figure(dfs, out_png="asr_vs_asri.png", out_pdf="asr_vs_asri.pdf"):
    fig, axes = plt.subplots(1, len(dfs), figsize=(9.2, 3.7), sharey=True)
    axes = np.atleast_1d(axes)
    for ax, (name, df) in zip(axes, dfs.items()):
        x = np.arange(len(df))
        ax.fill_between(x, df.ASR_I, df.ASR, color="tab:red", alpha=0.15,
                        label=r"Over-estimation (ASR$-$ASR-I)")
        ax.plot(x, df.ASR, "o--", color="tab:red", lw=1.8, label="ASR (conventional)")
        ax.plot(x, df.ASR_I, "s-", color="tab:blue", lw=1.8, label="ASR-I (proposed)")
        ax.set_xticks(x)
        ax.set_xticklabels([("0\n(no attack)" if e == 0 else f"{e:g}") for e in df.eps], fontsize=8)
        ax.set_title(name, fontsize=10)
        ax.set_xlabel(r"Perturbation budget $\epsilon$")
        ax.set_ylim(-0.03, 1.03); ax.grid(alpha=0.3)
        ax.scatter([0], [df.ASR.iloc[0]], s=60, facecolors="none", edgecolors="tab:red", zorder=5)
        ax.annotate("ASR > 0 with no attack\n(ASR-I = 0)", xy=(0, df.ASR.iloc[0]),
                    xytext=(0.30, 0.55), textcoords="axes fraction", fontsize=8,
                    arrowprops=dict(arrowstyle="->", color="gray", lw=1))
    axes[0].set_ylabel("Malicious evasion rate")
    axes[-1].legend(fontsize=8, loc="lower right", framealpha=0.95)
    fig.tight_layout()
    fig.savefig(out_png, dpi=160); fig.savefig(out_pdf)
    return fig


def emit_latex_table(dfs, out_tex="asr_vs_asri_table.tex"):
    L = [r"\begin{table}[t]", r"\centering", r"\footnotesize", r"\setlength{\tabcolsep}{4pt}",
         r"\caption{Conventional ASR vs.\ the proposed ASR-I (white-box PGD), measured from "
         r"M1's real predictions. At $\epsilon=0$ (no attack) ASR equals the clean miss rate "
         r"while ASR-I is exactly $0$; the over-estimation $\text{ASR}-\text{ASR-I}$ is largest "
         r"where the attack is weakest.}",
         r"\label{tab:asr_vs_asri}", r"\begin{tabular}{l|c|ccc}", r"\hline",
         r"Dataset & $\epsilon$ & ASR & ASR-I & ASR$-$ASR-I \\", r"\hline"]
    for name, df in dfs.items():
        for _, r in df.iterrows():
            e = "0 (none)" if r.eps == 0 else f"{r.eps:g}"
            L.append(f"{name} & {e} & {r.ASR:.4f} & {r.ASR_I:.4f} & {r.gap:.4f} " + r"\\")
        L.append(r"\hline")
    L += [r"\end{tabular}", r"\end{table}"]
    text = "\n".join(L)
    with open(out_tex, "w") as f:
        f.write(text)
    return text

In [ ]:
# === RUN: train + attack across datasets, collect real ASR / ASR-I ====
records = []
for name, cfg in DATASETS.items():
    if not os.path.exists(cfg["engine"]):
        print(f"[skip] {name}: engine not found -> {cfg['engine']}"); continue
    if not os.path.exists(cfg["csv"]):
        print(f"[skip] {name}: csv not found -> {cfg['csv']}"); continue
    print(f"\n=== {name} ===")
    eng = load_engine(cfg["engine"])
    X, y, mask_1d = prepare_dataset(eng, cfg)
    print(f"X={X.shape} | malicious={int(np.asarray(y).sum())} | perturbable feats={int(mask_1d.sum())}/{mask_1d.size}")
    for run in range(N_RUNS_FIG):
        seed = 42 + run
        for d in run_one(eng, X, y, mask_1d, seed):
            d.update(dataset=name, run=run + 1)
            records.append(d)
        print(f"  run {run+1}/{N_RUNS_FIG} done")

long_df = pd.DataFrame(records)
long_df.to_csv("asr_vs_asri_runs.csv", index=False)   # raw per-run numbers
print("\nSaved raw runs -> asr_vs_asri_runs.csv", long_df.shape)
long_df.head()

In [ ]:
# === Build the figure + LaTeX table from the REAL results =============
assert len(long_df), "long_df is empty -- check engine/CSV paths in the config cell."
dfs = build_dataframes(long_df)
fig = make_figure(dfs)
latex = emit_latex_table(dfs)

pd.set_option("display.float_format", lambda v: f"{v:.4f}")
for name, df in dfs.items():
    print(f"\n# {name}")
    print(df[["eps", "ASR", "ASR_I", "gap"]].to_string(index=False))
print("\nSaved: asr_vs_asri.pdf, asr_vs_asri.png, asr_vs_asri_table.tex")
print("\n" + latex)
plt.show()